# Modelagem de tópicos com LDA

[▶ Abrir este notebook no Google Colab](https://colab.research.google.com/github/lalvim/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_09/04_modelagem_de_topicos_e_tempo.ipynb)

No LDA, cada documento é uma **mistura de tópicos**, e tópicos são distribuições de palavras.
“Tópico” é um componente probabilístico, não sinônimo automático de tema teórico.
O número de tópicos e o pré-processamento condicionam o resultado.

![Documentos misturam tópicos, e tópicos distribuem probabilidades sobre palavras.](imagens/04_lda.svg)

In [ ]:
# @title Preparação do ambiente — execute esta célula no Google Colab
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

URL_REPOSITORIO = 'https://github.com/lalvim/disciplina_computacao_aplicada_humanidades_digitais.git'
REPOSITORIO = Path(
    "/content/disciplina_computacao_aplicada_humanidades_digitais"
)
PASTA_UNIDADE = REPOSITORIO / 'unidade_09'

try:
    import google.colab  # type: ignore  # noqa: F401
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

if EM_COLAB:
    if not (REPOSITORIO / ".git").exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "main",
                URL_REPOSITORIO,
                str(REPOSITORIO),
            ],
            check=True,
        )

    PACOTES_COLAB = [('sklearn', 'scikit-learn>=1.5,<2')]
    ausentes = [
        especificacao
        for modulo, especificacao in PACOTES_COLAB
        if importlib.util.find_spec(modulo) is None
    ]
    if ausentes:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", *ausentes],
            check=True,
        )

    os.chdir(PASTA_UNIDADE)
    print("Ambiente preparado em:", Path.cwd())
else:
    print("Ambiente local: nenhuma clonagem necessária.")

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
documentos=pd.read_csv("dados/documentos_temas.csv")
contagem=CountVectorizer(min_df=2)
Xcontagem=contagem.fit_transform(documentos["texto"])
lda=LatentDirichletAllocation(
    n_components=3, random_state=42, learning_method="batch", max_iter=80,
    doc_topic_prior=.1, topic_word_prior=.1
)
mistura=lda.fit_transform(Xcontagem)
termos=contagem.get_feature_names_out()
descricoes=[]
for t,pesos in enumerate(lda.components_):
    top=pesos.argsort()[-10:][::-1]
    descricoes.append({"topico":t,"palavras":"; ".join(termos[top])})
tabela_topicos=pd.DataFrame(descricoes)
display(tabela_topicos)

## 2. Misturas e nomeação

Nomear requer ler documentos com alta proporção e documentos mistos. Um rótulo
curto é interpretação provisória; mantenha também as palavras e exemplos que o
sustentam para que outra pessoa possa discordar.

In [ ]:
for t in range(3): documentos[f"topico_{t}"]=mistura[:,t]
documentos["topico_predominante"]=mistura.argmax(axis=1)
casos=[]
for t in range(3):
    amostra=documentos.nlargest(3,f"topico_{t}")[["id_documento","texto",f"topico_{t}"]].copy()
    amostra=amostra.rename(columns={f"topico_{t}":"proporcao_topico"})
    amostra["topico"]=t
    casos.append(amostra)
casos_topicos=pd.concat(casos,ignore_index=True)
display(casos_topicos[["topico","proporcao_topico","id_documento","texto"]])

## 3. Evolução temporal sem confundir composição e mudança histórica

Agregamos proporções médias por período. Uma curva pode mudar porque o corpus mudou,
porque palavras mudaram de sentido ou porque a simulação alterou a composição. O
gráfico gera perguntas para inspeção documental; não explica sozinho o processo.

In [ ]:
colunas_topicos=[f"topico_{t}" for t in range(3)]
evolucao=documentos.groupby("periodo")[colunas_topicos].mean()
display(evolucao.round(3))
evolucao.plot(marker="o",ylim=(0,1),title="Proporção média dos tópicos — dados fictícios")
plt.ylabel("Proporção média"); plt.xlabel("Período"); plt.tight_layout(); plt.show()

### U09-A05 — Atividade — tópicos como hipóteses

Nomeie cada tópico usando palavras e três documentos; identifique ao menos um caso
misto; escolha uma mudança temporal e formule duas explicações alternativas, uma
substantiva e outra ligada ao corpus. **Dinâmica (35 minutos):** grupos de três
defendem nomes diferentes e preservam a divergência no registro.

**Resposta:** escreva aqui.

## Passagem

A oficina exige escolher entre perfis numéricos, clusters textuais, tópicos ou não
adoção. Mais métodos não significam automaticamente mais evidência.